# 02. データ統合・クレンジング

staging の生データを統合・クレンジングし、builder.py 向け supplement JSON を生成する。

In [ ]:
DRY_RUN = False

import sys, json, logging
from pathlib import Path

try:
    BASE
except NameError:
    BASE        = Path("/content/AI_TradeManagement")
    STAGING_DIR = BASE / "data" / "staging"
    sys.path.insert(0, str(BASE / "scripts"))

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
print(f"DRY_RUN={DRY_RUN}")

## 1. 制裁リスト統合・クレンジング

In [ ]:
from pipeline.transform.sanctions_cleaner import clean_and_merge

ofac_path  = STAGING_DIR / "sanctions" / "ofac_sdn_raw.json"
bis_path   = STAGING_DIR / "sanctions" / "bis_el_raw.json"
merged_out = STAGING_DIR / "sanctions" / "sanctions_merged.json"

all_raw = []
for p in [ofac_path, bis_path]:
    if p.exists():
        data = json.loads(p.read_text())
        all_raw.extend(data)
        print(f"  Loaded: {p.name} ({len(data):,} 件)")
    else:
        print(f"  ⚠️  Not found: {p.name} — 01 を先に実行してください")

merged = clean_and_merge(all_raw, dry_run=DRY_RUN, output_path=merged_out)
print(f"\n✅ 統合後: {len(merged):,} 件")

## 2. FEFTA 欠損ノード補完 supplement 生成

In [ ]:
from pipeline.transform.fefta_enricher import build_fefta_supplement

fefta_articles_path = STAGING_DIR / "fefta" / "fefta_articles_raw.json"
fefta_out           = STAGING_DIR / "fefta" / "fefta_supplement.json"
control_nodes_path  = BASE / "data" / "unified" / "control_nodes.json"

if fefta_articles_path.exists():
    fefta_articles = json.loads(fefta_articles_path.read_text())
    supplement = build_fefta_supplement(
        control_nodes_path=control_nodes_path,
        fefta_articles=fefta_articles,
        output_path=fefta_out,
        dry_run=DRY_RUN,
    )
    print(f"✅ FEFTA supplement: {len(supplement)} ノード補完")
else:
    print("⚠️  fefta_articles_raw.json なし — 01 を先に実行してください")
    supplement = {}

## 3. サマリー

In [ ]:
nodes = json.loads(control_nodes_path.read_text())["nodes"]
missing_fefta = [n for n in nodes if n.get("regime") == "fefta" and not n.get("requirement_text")]
missing_eccn  = [n for n in nodes if n.get("regime") == "ear"   and not n.get("requirement_text")]
print(f"制裁エンティティ (統合後): {len(merged):>6,} 件")
print(f"FEFTA 補完生成          : {len(supplement):>6} ノード")
print(f"FEFTA 欠損残り          : {len(missing_fefta):>6} ノード")
print(f"ECCN  欠損残り          : {len(missing_eccn):>6} ノード")
print("\n次のノートブック → 03_build_mappings.ipynb")